# 02 - Preprocessing: Splits, Vocabulários e Matriz de Interações

Gera, uma única vez, os artefatos consumidos por todos os notebooks de modelo (03 a 07): vocabulários de usuário/item, a matriz esparsa de histórico de compras (`prior`) e os splits de treino/validação/teste.

Ver `docs/NOTEBOOKS.md` (seção 2) para o racional do split e do corte de catálogo top-N.

## 0. Configuração Inicial

In [1]:
import hashlib
import json
import pickle
import random
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd
import scipy.sparse as sp
import yaml
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42


def set_seed(seed: int) -> None:
    """Fixa a seed de aleatoriedade para reprodutibilidade."""
    random.seed(seed)
    np.random.seed(seed)


set_seed(RANDOM_SEED)

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
CONFIG_PATH = Path("../configs/model_config.yaml")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

with open(CONFIG_PATH, encoding="utf-8") as f:
    config = yaml.safe_load(f)

TOP_N_PRODUCTS = config["preprocessing"]["top_n_products"]
TRAIN_RATIO = config["preprocessing"]["train_ratio"]
VAL_RATIO = config["preprocessing"]["val_ratio"]
TEST_RATIO = config["preprocessing"]["test_ratio"]

mlflow.set_tracking_uri(f"file:{(Path('..') / 'mlruns').resolve()}")
mlflow.set_experiment("preprocessing")

print(f"top_n_products={TOP_N_PRODUCTS}, split={TRAIN_RATIO}/{VAL_RATIO}/{TEST_RATIO}")

top_n_products=3000, split=0.7/0.15/0.15


## 1. Carregamento dos Dados Brutos

In [2]:
RAW_FILES = [
    RAW_DIR / "orders.csv",
    RAW_DIR / "order_products__prior.csv",
    RAW_DIR / "order_products__train.csv",
    RAW_DIR / "products.csv",
]

orders = pd.read_csv(
    RAW_FILES[0],
    dtype={"order_id": "int32", "user_id": "int32", "eval_set": "category"},
    usecols=["order_id", "user_id", "eval_set"],
)
order_products_prior = pd.read_csv(
    RAW_FILES[1],
    dtype={"order_id": "int32", "product_id": "int32"},
    usecols=["order_id", "product_id"],
)
order_products_train = pd.read_csv(
    RAW_FILES[2],
    dtype={"order_id": "int32", "product_id": "int32"},
    usecols=["order_id", "product_id"],
)
products = pd.read_csv(RAW_FILES[3], dtype={"product_id": "int32"})

n_orders = len(orders)
n_prior = len(order_products_prior)
n_train_rows = len(order_products_train)
print(f"orders: {n_orders:,} | prior: {n_prior:,} | train: {n_train_rows:,}")

orders: 3,421,083 | prior: 32,434,489 | train: 1,384,617


## 2. Catálogo Top-N (Escopo de Produtos)

In [3]:
product_counts = order_products_prior["product_id"].value_counts()
top_products = product_counts.head(TOP_N_PRODUCTS).index
product_id_to_idx = {pid: idx for idx, pid in enumerate(top_products)}
idx_to_product_id = np.array(top_products, dtype=np.int32)

coverage = product_counts.head(TOP_N_PRODUCTS).sum() / product_counts.sum()
print(f"Top-{TOP_N_PRODUCTS} produtos cobrem {coverage:.1%} do volume de compras")

Top-3000 produtos cobrem 73.1% do volume de compras


## 3. Cesto-Rótulo (`eval_set == "train"`) Restrito ao Catálogo

In [4]:
order_to_user_train = orders.loc[orders["eval_set"] == "train"].set_index("order_id")[
    "user_id"
]

train_eval_products = order_products_train[
    order_products_train["product_id"].isin(product_id_to_idx)
].copy()
train_eval_products["user_id"] = train_eval_products["order_id"].map(
    order_to_user_train
)

future_baskets = train_eval_products.groupby("user_id")["product_id"].apply(set)
eval_users = future_baskets.index.to_numpy()

print(f"Usuários com cesto-rótulo dentro do catálogo top-N: {len(eval_users):,}")

Usuários com cesto-rótulo dentro do catálogo top-N: 126,408


## 4. Segmentação de Usuários (para Split Estratificado)

Mesma lógica de segmentação aplicada em `01_eda.ipynb`, recalculada aqui sobre o total de pedidos de cada usuário (todos os `eval_set`).

In [5]:
def segment_users(orders_per_user: pd.Series) -> pd.Series:
    """Segmenta usuários por número total de pedidos.

    Args:
        orders_per_user: Série com a contagem de pedidos por usuário.

    Returns:
        Série categórica com o segmento de cada usuário.
    """
    bins = [0, 3, 10, orders_per_user.max()]
    labels = ["ocasional", "regular", "super_user"]
    return pd.cut(orders_per_user, bins=bins, labels=labels, include_lowest=True)


orders_per_user = orders.groupby("user_id", observed=True).size()
user_segments = segment_users(orders_per_user)
eval_user_segments = user_segments.loc[eval_users]
eval_user_segments.value_counts()

regular       63654
super_user    62754
ocasional         0
Name: count, dtype: int64

## 5. Split Estratificado Treino/Validação/Teste

In [6]:
train_users, rest_users = train_test_split(
    eval_users,
    train_size=TRAIN_RATIO,
    random_state=RANDOM_SEED,
    stratify=eval_user_segments,
)
val_users, test_users = train_test_split(
    rest_users,
    train_size=VAL_RATIO / (VAL_RATIO + TEST_RATIO),
    random_state=RANDOM_SEED,
    stratify=user_segments.loc[rest_users],
)

print(f"train={len(train_users):,} | val={len(val_users):,} | test={len(test_users):,}")

train=88,485 | val=18,961 | test=18,962


## 6. Vocabulário de Usuários e Matriz Esparsa de Interações (`prior`)

In [7]:
idx_to_user_id = np.sort(eval_users).astype(np.int32)
user_id_to_idx = {uid: idx for idx, uid in enumerate(idx_to_user_id)}

order_to_user_prior = orders.loc[orders["eval_set"] == "prior"].set_index("order_id")[
    "user_id"
]

prior_filtered = order_products_prior[
    order_products_prior["product_id"].isin(product_id_to_idx)
].copy()
prior_filtered["user_id"] = prior_filtered["order_id"].map(order_to_user_prior)
prior_filtered = prior_filtered[prior_filtered["user_id"].isin(user_id_to_idx)]

n_prior_filtered = len(prior_filtered)
print(f"Interações 'prior' filtradas (catálogo/usuários): {n_prior_filtered:,}")

Interações 'prior' filtradas (catálogo/usuários): 14,914,655


In [8]:
def build_interaction_matrix(
    interactions: pd.DataFrame,
    user_id_to_idx: dict,
    product_id_to_idx: dict,
    n_users: int,
    n_items: int,
) -> sp.csr_matrix:
    """Constrói a matriz esparsa binária usuário-item a partir de interações.

    Args:
        interactions: DataFrame com colunas user_id e product_id.
        user_id_to_idx: Mapeamento de user_id para índice de linha.
        product_id_to_idx: Mapeamento de product_id para índice de coluna.
        n_users: Número total de usuários no vocabulário.
        n_items: Número total de itens no vocabulário.

    Returns:
        Matriz esparsa CSR binária (n_users, n_items).
    """
    rows = interactions["user_id"].map(user_id_to_idx).to_numpy()
    cols = interactions["product_id"].map(product_id_to_idx).to_numpy()
    data = np.ones(len(rows), dtype=np.int8)
    matrix = sp.coo_matrix((data, (rows, cols)), shape=(n_users, n_items)).tocsr()
    return (matrix > 0).astype(np.int8)


n_users = len(idx_to_user_id)
n_items = len(idx_to_product_id)
interactions_prior = build_interaction_matrix(
    prior_filtered, user_id_to_idx, product_id_to_idx, n_users, n_items
)

density = interactions_prior.nnz / (n_users * n_items)
print(f"Matriz de interações: {interactions_prior.shape}, densidade={density:.4%}")

Matriz de interações: (126408, 3000), densidade=1.4479%


## 7. Pares Positivos por Split

In [9]:
def build_pairs(
    users: np.ndarray,
    future_baskets: pd.Series,
    user_id_to_idx: dict,
    product_id_to_idx: dict,
) -> pd.DataFrame:
    """Constrói os pares positivos (user_idx, item_idx) de um split.

    Args:
        users: Lista de user_ids pertencentes ao split.
        future_baskets: Série indexada por user_id com o set de product_ids
            do cesto futuro.
        user_id_to_idx: Mapeamento de user_id para índice.
        product_id_to_idx: Mapeamento de product_id para índice.

    Returns:
        DataFrame com colunas user_idx e item_idx, um par positivo por linha.
    """
    pairs = [
        (user_id_to_idx[uid], product_id_to_idx[pid])
        for uid in users
        for pid in future_baskets[uid]
    ]
    return pd.DataFrame(pairs, columns=["user_idx", "item_idx"])


train_pairs = build_pairs(
    train_users, future_baskets, user_id_to_idx, product_id_to_idx
)
val_pairs = build_pairs(val_users, future_baskets, user_id_to_idx, product_id_to_idx)
test_pairs = build_pairs(test_users, future_baskets, user_id_to_idx, product_id_to_idx)

n_train_pairs, n_val_pairs, n_test_pairs = (
    len(train_pairs),
    len(val_pairs),
    len(test_pairs),
)
print(f"pares: train={n_train_pairs:,} | val={n_val_pairs:,} | test={n_test_pairs:,}")

pares: train=689,910 | val=147,983 | test=146,770


## 8. Persistência dos Artefatos

In [10]:
def compute_dataset_hash(paths: list) -> str:
    """Calcula o hash SHA256 combinado dos arquivos de dados brutos usados.

    Args:
        paths: Lista de caminhos de arquivos a incluir no hash.

    Returns:
        Hash SHA256 hexadecimal combinado dos arquivos.
    """
    hasher = hashlib.sha256()
    for path in sorted(paths):
        with open(path, "rb") as f:
            for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
                hasher.update(chunk)
    return hasher.hexdigest()


idx_to_product_name = (
    products.set_index("product_id").loc[idx_to_product_id, "product_name"].to_numpy()
)

vocabularies = {
    "user_id_to_idx": user_id_to_idx,
    "idx_to_user_id": idx_to_user_id,
    "product_id_to_idx": product_id_to_idx,
    "idx_to_product_id": idx_to_product_id,
    "idx_to_product_name": idx_to_product_name,
}

with open(PROCESSED_DIR / "vocabularies.pkl", "wb") as f:
    pickle.dump(vocabularies, f)

sp.save_npz(PROCESSED_DIR / "interactions_prior.npz", interactions_prior)
train_pairs.to_pickle(PROCESSED_DIR / "train_pairs.pkl")
val_pairs.to_pickle(PROCESSED_DIR / "val_pairs.pkl")
test_pairs.to_pickle(PROCESSED_DIR / "test_pairs.pkl")

dataset_hash = compute_dataset_hash(RAW_FILES)
split_meta = {
    "random_seed": RANDOM_SEED,
    "top_n_products": TOP_N_PRODUCTS,
    "train_ratio": TRAIN_RATIO,
    "val_ratio": VAL_RATIO,
    "test_ratio": TEST_RATIO,
    "n_users_train": len(train_users),
    "n_users_val": len(val_users),
    "n_users_test": len(test_users),
    "n_items_catalog": n_items,
    "catalog_coverage": float(coverage),
    "dataset_hash": dataset_hash,
}
with open(PROCESSED_DIR / "split_meta.json", "w", encoding="utf-8") as f:
    json.dump(split_meta, f, indent=2)

split_meta

{'random_seed': 42,
 'top_n_products': 3000,
 'train_ratio': 0.7,
 'val_ratio': 0.15,
 'test_ratio': 0.15,
 'n_users_train': 88485,
 'n_users_val': 18961,
 'n_users_test': 18962,
 'n_items_catalog': 3000,
 'catalog_coverage': 0.7312173162339631,
 'dataset_hash': 'c3fda3f4a8d64ffdd01d4cf3bebc2ebc9385414135251f90438e2b7b72645e8a'}

## 9. Rastreamento no MLflow

In [11]:
with mlflow.start_run(run_name="preprocessing_v1"):
    mlflow.log_params(
        {
            "top_n_products": TOP_N_PRODUCTS,
            "train_ratio": TRAIN_RATIO,
            "val_ratio": VAL_RATIO,
            "test_ratio": TEST_RATIO,
            "random_seed": RANDOM_SEED,
            "dataset_hash": dataset_hash,
        }
    )
    mlflow.log_metrics(
        {
            "n_users_train": len(train_users),
            "n_users_val": len(val_users),
            "n_users_test": len(test_users),
            "n_items_catalog": n_items,
            "catalog_coverage": float(coverage),
        }
    )
    mlflow.log_artifact(str(PROCESSED_DIR / "split_meta.json"))

print("Preprocessing concluído e rastreado no MLflow.")

Preprocessing concluído e rastreado no MLflow.
